# Treinamento parcial U-Mamba (estágio 12)

Executa um experimento curto da arquitetura oficial U-Mamba adaptada ao dataset RGB atual. O notebook prepara o ambiente automaticamente caso seja aberto em um novo runtime.

**Antes de executar:** selecione uma GPU NVIDIA em `Ambiente de execução → Alterar tipo de ambiente de execução`.

## Bootstrap e preparação do ambiente

Carrega o código atual do TCC, exige GPU CUDA e prepara Mamba + a arquitetura oficial fixada no mesmo commit usado pelo notebook 11.

In [ ]:
import importlib
import json
import pathlib
import sys
import urllib.request

BOOTSTRAP_URL = "https://raw.githubusercontent.com/oguel/tcc-umamba/main/src/bootstrap.py"
pathlib.Path("bootstrap.py").write_bytes(urllib.request.urlopen(BOOTSTRAP_URL).read())
sys.path.insert(0, str(pathlib.Path.cwd()))
bootstrap = importlib.import_module("bootstrap")
importlib.reload(bootstrap)
workspace = bootstrap.bootstrap_workspace()

from src.models.umamba_runtime import ensure_umamba_runtime

runtime_info = ensure_umamba_runtime()
print(json.dumps(runtime_info, indent=2, ensure_ascii=False))


## Imports e dispositivo

Prepara o conjunto experimental, loss, métricas, otimizador e GPU.

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader

from src import io
from src.config import get_config
from src.data.dataset import CoffeeSegmentationDataset
from src.losses import BCEDiceLoss
from src.models.umamba import build_official_umamba_enc_2d
from src.trainer import fit_model, run_epoch
from src.utils import set_all_seeds

config = get_config()
set_all_seeds(int(config["reproducibility"]["seed"]))
device = torch.device("cuda")
print(f"GPU: {torch.cuda.get_device_name(0)}")


## DataLoaders

Reutiliza exatamente os mesmos subconjuntos do baseline para manter a comparação preliminar coerente.

In [ ]:
storage_paths = io.resolve_storage_paths()
dataset_root = storage_paths["data_embrapa"]
image_size = int(config["data"]["patch_size"])

train_dataset = CoffeeSegmentationDataset(dataset_root / "Imagens_treino", dataset_root / "Mascaras_treino", image_size=image_size, augment=True)
val_dataset = CoffeeSegmentationDataset(dataset_root / "Imagens_validacao", dataset_root / "Mascaras_validacao", image_size=image_size)
test_dataset = CoffeeSegmentationDataset(dataset_root / "Imagens_teste", dataset_root / "Mascaras_teste", image_size=image_size)

batch_size = int(config["training"].get("umamba_batch_size", 1))
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=True)
print(f"Treino: {len(train_dataset)} | Validação: {len(val_dataset)} | Teste: {len(test_dataset)}")


## Construção da U-Mamba

Instancia a classe oficial `UMambaEnc_2d` com configuração reduzida para o experimento parcial RGB.

In [ ]:
features = tuple(int(value) for value in config["model"]["umamba_features"])
model = build_official_umamba_enc_2d(
    input_channels=int(config["model"]["input_channels_experimental"]),
    num_classes=int(config["model"]["output_channels"]),
    input_size=(image_size, image_size),
    features_per_stage=features,
).to(device)

criterion = BCEDiceLoss(
    bce_weight=float(config["loss"]["bce_weight"]),
    dice_weight=float(config["loss"]["dice_weight"]),
)
optimizer = AdamW(model.parameters(), lr=float(config["training"]["lr"]), weight_decay=1e-4)
parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(f"Parâmetros: {parameter_count:,}")


## Treinamento curto

Executa poucas épocas apenas para verificar estabilidade do treinamento, uso de memória e capacidade de produzir segmentações.

In [ ]:
epochs = int(config["training"].get("umamba_partial_epochs", 3))
checkpoint_path = storage_paths["models_umamba"] / "best_umamba_experimental.pt"

history = fit_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    epochs=epochs,
    checkpoint_path=checkpoint_path,
)


## Métricas e eficiência

Salva o histórico de loss, IoU, F1, tempo por época e pico de VRAM e avalia o melhor checkpoint no teste.

In [ ]:
history_df = pd.DataFrame(history)
metrics_dir = storage_paths["artifacts_metrics"] / "umamba"
figures_dir = storage_paths["artifacts_figures"] / "umamba"
metrics_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

history_path = metrics_dir / "experimental_history.csv"
history_df.to_csv(history_path, index=False)

checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
test_metrics = run_epoch(model, test_loader, criterion, device)
test_metrics.update({
    "best_epoch": int(checkpoint["epoch"]),
    "best_val_iou": float(checkpoint["val_iou"]),
    "parameters": int(parameter_count),
})
metrics_path = metrics_dir / "experimental_test_metrics.json"
metrics_path.write_text(json.dumps(test_metrics, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(test_metrics, indent=2, ensure_ascii=False))


## Previsões qualitativas

Produz uma amostra visual preliminar da U-Mamba no conjunto de teste.

In [ ]:
model.eval()
sample_count = min(3, len(test_dataset))
figure, axes = plt.subplots(sample_count, 3, figsize=(12, 4 * sample_count))
if sample_count == 1:
    axes = axes.reshape(1, -1)

with torch.inference_mode():
    for row in range(sample_count):
        image, mask = test_dataset[row]
        logits = model(image.unsqueeze(0).to(device))
        prediction = (torch.sigmoid(logits)[0, 0] >= 0.5).cpu().numpy()
        axes[row, 0].imshow(image.permute(1, 2, 0).numpy())
        axes[row, 0].set_title("Imagem")
        axes[row, 1].imshow(mask[0].numpy(), cmap="gray", vmin=0, vmax=1)
        axes[row, 1].set_title("Ground Truth")
        axes[row, 2].imshow(prediction, cmap="gray", vmin=0, vmax=1)
        axes[row, 2].set_title("Predição U-Mamba")
        for col in range(3):
            axes[row, col].axis("off")

figure.tight_layout()
prediction_path = figures_dir / "experimental_test_predictions.png"
figure.savefig(prediction_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figura salva em: {prediction_path}")


## Limite científico deste experimento

O resultado é apenas uma prova de conceito sobre o dataset RGB atual. Ele não substitui o treinamento definitivo em imagens Sentinel-2 multiespectrais nem deve ser usado para concluir que U-Mamba supera ou não U-Net.